In [1]:
print("Hello, World!")

Hello, World!


In [ ]:
import os
import json
import base64
import requests
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Configuration
PDF_FOLDER = '/home/ttran/code/ai_pdf_reader/data'
OLLAMA_URL = 'http://ollamacuda.home:11434/api/chat'
MODEL_NAME = 'qwen3-vl:8b'  # Vision model; Ollama passes the PDF to it natively
MAX_WORKERS = 4  # Limit parallelism to avoid overloading the Ollama server
KEYS_TO_EXTRACT = [
    'Date of Sale',
    'Earliest Time Sale Will Begin',
    'Place of Sale of Property',
    'Legal Description',
    'Date Added',
    'Physical Address'
]

PROMPT_TEMPLATE = """You are extracting data from a property sale notice PDF.
Extract the following fields from the document:
{keys}

Rules:
- "Physical Address" is the property's physical/street address (sometimes labeled "commonly known as").
- Return ONLY a valid JSON object with exactly these keys: {key_list}.
- If a field is not present in the document, set its value to null.
- Do not include any text outside the JSON object.
"""

def query_ollama(pdf_path: str) -> dict:
    """Send the raw PDF to Ollama's chat API and get back extracted JSON."""
    with open(pdf_path, 'rb') as f:
        pdf_b64 = base64.b64encode(f.read()).decode('utf-8')

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": PROMPT_TEMPLATE.format(
                    keys="\n".join(f"- {k}" for k in KEYS_TO_EXTRACT),
                    key_list=json.dumps(KEYS_TO_EXTRACT),
                ),
                # Ollama natively ingests PDFs for multimodal models
                "images": [pdf_b64],
            }
        ],
        "stream": False,
        "format": "json",
        "options": {"temperature": 0},
    }

    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=300)
        response.raise_for_status()
        content = response.json()['message']['content']
        return json.loads(content)
    except Exception as e:
        print(f"Error querying Ollama for {Path(pdf_path).name}: {e}")
        return {}

def process_pdf(file_path: str) -> dict:
    """Workflow for a single PDF file: file name (no extension) -> extracted fields."""
    file_name = Path(file_path).stem
    extracted_data = query_ollama(file_path)

    # Normalize: guarantee all keys exist, with null for missing ones
    return {file_name: {k: extracted_data.get(k) for k in KEYS_TO_EXTRACT}}

def main() -> list:
    pdf_files = sorted(
        os.path.join(PDF_FOLDER, f)
        for f in os.listdir(PDF_FOLDER)
        if f.lower().endswith('.pdf')
    )
    print(f"Processing {len(pdf_files)} PDFs...")

    results = []
    # Parallel processing: API calls are I/O bound, threads work well here
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_pdf, pdf): pdf for pdf in pdf_files}
        for future in futures:
            try:
                results.append(future.result())
            except Exception as e:
                print(f"Task failed for {futures[future]}: {e}")

    return results

if __name__ == "__main__":
    final_results = main()
    print(json.dumps(final_results, indent=2))


Error querying Ollama: 503 Server Error: Service Unavailable for url: http://192.168.10.12:11434/api/generate
Error querying Ollama: 503 Server Error: Service Unavailable for url: http://192.168.10.12:11434/api/generate
Error querying Ollama: 503 Server Error: Service Unavailable for url: http://192.168.10.12:11434/api/generate
Error querying Ollama: 503 Server Error: Service Unavailable for url: http://192.168.10.12:11434/api/generate
Error querying Ollama: 503 Server Error: Service Unavailable for url: http://192.168.10.12:11434/api/generate
Error querying Ollama: 503 Server Error: Service Unavailable for url: http://192.168.10.12:11434/api/generate
Error querying Ollama: 503 Server Error: Service Unavailable for url: http://192.168.10.12:11434/api/generate
Error querying Ollama: 503 Server Error: Service Unavailable for url: http://192.168.10.12:11434/api/generate
Error querying Ollama: 503 Server Error: Service Unavailable for url: http://192.168.10.12:11434/api/generate
Error quer